# 01 Scraping
Scrape FOMC statements and minutes from the Federal Reserve website.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
import re
import time
from urllib.parse import urljoin

BASE_URL = 'https://www.federalreserve.gov'
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; ResearchBot/1.0)'}
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def fetch(url, sleep=1.0):
    # rate-limited GET
    time.sleep(sleep)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

In [3]:
def extract_statement_links(html, page_url):
    # statement URL pattern: /newsevents/pressreleases/monetaryYYYYMMDDa.htm
    # or for historical pages, similar pattern
    soup = BeautifulSoup(html, 'lxml')
    results = []
    # find all <a> tags whose href matches the monetary statement pattern
    pattern = re.compile(r'/newsevents/pressreleases/monetary(\d{8})a?\.htm', re.IGNORECASE)
    for a in soup.find_all('a', href=True):
        m = pattern.search(a['href'])
        if not m:
            continue
        date_str = m.group(1)  # YYYYMMDD
        full_url = urljoin(BASE_URL, a['href'])
        link_text = a.get_text(strip=True).lower()
        # filter: we want the "statement" link, not "minutes" or "implementation note"
        # rely on URL pattern (monetary...a.htm = statement)
        results.append({'date_str': date_str, 'url': full_url, 'link_text': link_text})
    return results

In [4]:
current_url = f'{BASE_URL}/monetarypolicy/fomccalendars.htm'
html = fetch(current_url)
current_links = extract_statement_links(html, current_url)
print(f'found {len(current_links)} links on current page')
pd.DataFrame(current_links).head(10)

found 44 links on current page


,date_str,url,link_text
0,20260128,https://www.federalreserve.gov/newsevents/pres...,html
1,20260318,https://www.federalreserve.gov/newsevents/pres...,html
2,20260429,https://www.federalreserve.gov/newsevents/pres...,html
3,20250129,https://www.federalreserve.gov/newsevents/pres...,html
4,20250319,https://www.federalreserve.gov/newsevents/pres...,html
5,20250507,https://www.federalreserve.gov/newsevents/pres...,html
6,20250618,https://www.federalreserve.gov/newsevents/pres...,html
7,20250730,https://www.federalreserve.gov/newsevents/pres...,html
8,20250822,https://www.federalreserve.gov/newsevents/pres...,statement on longer-run goals and monetary pol...
9,20250917,https://www.federalreserve.gov/newsevents/pres...,html


In [5]:
historical_urls = [f'{BASE_URL}/monetarypolicy/fomchistorical{y}.htm' for y in range(2000, 2021)]
all_links = list(current_links)
for url in historical_urls:
    try:
        html = fetch(url)
        links = extract_statement_links(html, url)
        print(f'{url.split("/")[-1]}: {len(links)} links')
        all_links.extend(links)
    except Exception as e:
        print(f'FAILED {url}: {e}')
print(f'total: {len(all_links)}')

fomchistorical2000.htm: 0 links
fomchistorical2001.htm: 0 links
fomchistorical2002.htm: 0 links
fomchistorical2003.htm: 0 links
fomchistorical2004.htm: 0 links
fomchistorical2005.htm: 0 links
fomchistorical2006.htm: 0 links
fomchistorical2007.htm: 0 links
fomchistorical2008.htm: 0 links
fomchistorical2009.htm: 0 links
fomchistorical2010.htm: 0 links
fomchistorical2011.htm: 8 links
fomchistorical2012.htm: 8 links
fomchistorical2013.htm: 8 links
fomchistorical2014.htm: 8 links
fomchistorical2015.htm: 8 links
fomchistorical2016.htm: 8 links
fomchistorical2017.htm: 8 links
fomchistorical2018.htm: 8 links
fomchistorical2019.htm: 9 links
fomchistorical2020.htm: 12 links
total: 129


In [6]:
df = pd.DataFrame(all_links).drop_duplicates(subset=['url']).reset_index(drop=True)
df['date'] = pd.to_datetime(df['date_str'], format='%Y%m%d')
df = df.sort_values('date').reset_index(drop=True)
df = df[['date', 'url', 'link_text']]
print(df.shape)
print(df['date'].min(), '→', df['date'].max())
df.head()

(129, 3)
2011-01-26 00:00:00 → 2026-04-29 00:00:00


,date,url,link_text
0,2011-01-26,https://www.federalreserve.gov/newsevents/pres...,statement
1,2011-03-15,https://www.federalreserve.gov/newsevents/pres...,statement
2,2011-04-27,https://www.federalreserve.gov/newsevents/pres...,statement
3,2011-06-22,https://www.federalreserve.gov/newsevents/pres...,statement
4,2011-08-09,https://www.federalreserve.gov/newsevents/pres...,statement


In [7]:
df['year'] = df['date'].dt.year
df.groupby('year').size()

year
2011     8
2012     8
2013     8
2014     8
2015     8
2016     8
2017     8
2018     8
2019     9
2020    12
2021     8
2022     8
2023     8
2024     8
2025     9
2026     3
dtype: int64

In [8]:
df.drop(columns=['year']).to_csv(PROCESSED_DIR / 'fomc_urls.csv', index=False)
print('saved:', PROCESSED_DIR / 'fomc_urls.csv')

saved: ../data/processed/fomc_urls.csv
